# Extratos Bancários

Ingestão e limpeza dos extratos mensais exportados do sistema **ClienteOnline** (imobiliária).

- **Fonte**: `exports/hojas/extrato/` — 11 arquivos `.xls` (ago/2025 – jun/2026)
- **Formato**: HTML-as-XLS com múltiplas subcontas por arquivo
- **Saída**: `exports/csv/extratos.csv`

In [2]:
%pip install -q lxml openpyxl matplotlib seaborn pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
import re
from pathlib import Path

import lxml.html
import pandas as pd

In [4]:
def parse_br_number(s):
    """Converte string no formato brasileiro (1.234,56 ou -1.234,56) para float."""
    if isinstance(s, (int, float)):
        return float(s) if not pd.isna(s) else None
    if not isinstance(s, str):
        return None
    s = s.strip()
    if s in ("-", "", "nan", "None"):
        return None
    s = re.sub(r"\s", "", s)
    s = s.replace(".", "").replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return None


def parse_extrato_file(filepath: Path) -> pd.DataFrame:
    """
    Parseia um arquivo .xls (HTML-as-XLS) de extrato do sistema ClienteOnline.

    Cada arquivo contém múltiplas subcontas separadas por linhas de cabeçalho.
    Retorna DataFrame com: mes_ano, data, subconta, historico, complemento,
    debito, credito, saldo, saldo_anterior.
    """
    # Inferir mes_ano do nome do arquivo: "25_11.xls" → "2025-11"
    parts = filepath.stem.split("_")
    mes_ano = f"20{parts[0]}-{parts[1].zfill(2)}"

    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        html = f.read()

    tree = lxml.html.fromstring(html)
    rows = tree.xpath("//tr")

    records = []
    current_subconta = None
    saldo_anterior = None
    date_re = re.compile(r"^\d{2}/\d{2}/\d{2,4}$")

    for row in rows:
        cells = row.xpath(".//td")
        if not cells:
            continue

        first_cell = cells[0]
        first_text = first_cell.text_content().strip()
        first_colspan = int(first_cell.get("colspan", 1))

        # Linha de cabeçalho de subconta: 1 célula com colspan ≥ 7
        if len(cells) == 1 and first_colspan >= 7:
            current_subconta = re.sub(r"^\d+\s*-\s*", "", first_text).strip()
            saldo_anterior = None
            continue

        # Linha de cabeçalho de colunas: primeira célula = "Data"
        if first_text == "Data":
            continue

        # Linha SALDO ANTERIOR: colspan ≥ 5 e texto contém "SALDO ANTERIOR"
        if "SALDO ANTERIOR" in first_text and first_colspan >= 5:
            saldo_text = cells[1].text_content().strip() if len(cells) > 1 else ""
            saldo_anterior = parse_br_number(saldo_text)
            continue

        # Linha de transação: primeira célula é uma data
        if date_re.match(first_text) and len(cells) >= 7:
            texts = [c.text_content().strip() for c in cells]
            records.append(
                {
                    "mes_ano": mes_ano,
                    "data": texts[0],
                    "subconta": current_subconta,
                    "historico": texts[2],
                    "complemento": texts[3],
                    "debito": parse_br_number(texts[4]),
                    "credito": parse_br_number(texts[5]),
                    "saldo": parse_br_number(texts[6]),
                    "saldo_anterior": saldo_anterior,
                }
            )

    return pd.DataFrame(records)

In [5]:
EXTRATO_DIR = Path("../exports/hojas/extrato")
files = sorted(EXTRATO_DIR.glob("*.xls"))
print(f"Arquivos encontrados: {len(files)}\n")

dfs = []
for f in files:
    df = parse_extrato_file(f)
    print(f"  {f.name}: {len(df)} registros")
    dfs.append(df)

df_extratos = pd.concat(dfs, ignore_index=True)
print(f"\nTotal bruto: {len(df_extratos)} registros")
df_extratos.head()

Arquivos encontrados: 11

  25_08.xls: 396 registros
  25_09.xls: 535 registros
  25_10.xls: 520 registros
  25_11.xls: 558 registros
  25_12.xls: 553 registros
  26_01.xls: 562 registros
  26_02.xls: 520 registros
  26_03.xls: 694 registros
  26_04.xls: 747 registros
  26_05.xls: 583 registros
  26_06.xls: 547 registros

Total bruto: 6215 registros


,mes_ano,data,subconta,historico,complemento,debito,credito,saldo,saldo_anterior
0,2025-08,01/08/25,CONTA NORMAL,PG.TX.FIN.SD.DEV.07/2025,,-1552.0,NaN,-64023.24,-62471.24
1,2025-08,01/08/25,CONTA NORMAL,REC.MULTA+C.M.+JRS.,05/25 Apto 1008 BL B,NaN,11.04,-64012.20,-62471.24
2,2025-08,01/08/25,CONTA NORMAL,REC.CONDOMÍNIO,05/25 Apto 1008 BL B,NaN,269.05,-63743.15,-62471.24
3,2025-08,01/08/25,CONTA NORMAL,REC.CONDOMÍNIO,08/25 Apto 0408 BL B,NaN,316.10,-63427.05,-62471.24
4,2025-08,01/08/25,CONTA NORMAL,REC.CONDOMÍNIO,08/25 Apto 0801 BL B,NaN,320.80,-63106.25,-62471.24


In [7]:
# Conversão de tipos e limpeza
df_extratos["data"] = pd.to_datetime(df_extratos["data"], format="%d/%m/%y", errors="coerce")
df_extratos["historico"]   = df_extratos["historico"].str.strip()
df_extratos["complemento"] = df_extratos["complemento"].str.strip()
df_extratos["subconta"]    = df_extratos["subconta"].str.strip()

# Remover duplicatas exatas (mesmo lançamento pode aparecer em meses sobrepostos)
before = len(df_extratos)
df_extratos = df_extratos.drop_duplicates(
    subset=["data", "subconta", "historico", "complemento", "debito", "credito"]
)
print(f"Duplicatas removidas: {before - len(df_extratos)}")
print(f"Período: {df_extratos['data'].min().date()} → {df_extratos['data'].max().date()}")
print(f"Total final: {len(df_extratos)} registros\n")
df_extratos.info()

Duplicatas removidas: 0
Período: 2025-08-01 → 2026-06-30
Total final: 6215 registros

<class 'pandas.DataFrame'>
RangeIndex: 6215 entries, 0 to 6214
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   mes_ano         6215 non-null   str           
 1   data            6215 non-null   datetime64[us]
 2   subconta        6215 non-null   str           
 3   historico       6215 non-null   str           
 4   complemento     6215 non-null   str           
 5   debito          495 non-null    float64       
 6   credito         5720 non-null   float64       
 7   saldo           6215 non-null   float64       
 8   saldo_anterior  6215 non-null   float64       
dtypes: datetime64[us](1), float64(4), str(4)
memory usage: 437.1 KB


In [8]:
print("=== Subcontas ===")
print(df_extratos["subconta"].value_counts().to_string())

print("\n=== Top 25 Históricos ===")
print(df_extratos["historico"].value_counts().head(25).to_string())

print("\n=== Registros por mês ===")
print(df_extratos.groupby("mes_ano").size().to_string())

=== Subcontas ===
subconta
CONTA NORMAL         2422
FUNDO OBRAS          1745
FUNDO FÉRIAS          792
FUNDO DIF.SALÁRIO     630
FUNDO 13º SALÁRIO     479
USO DO BOX            119
ACORDOS                15
FUNDO INDEN.TRAB.       6
SAQUES P/ACERTO         5
FUNDO FÉRIAS/13º        2

=== Top 25 Históricos ===
historico
REC.CONDOMÍNIO                 1734
FUNDO OBRAS                    1734
REC.CH.EXTRA FÉRIAS             786
DIF.SALARIAL                    625
13º SALARIO                     474
REC. MULTA                      127
REC.ALUG.DEPÓSITO                84
REC.MULTA+C.M.+JRS.              83
REC.TAXA USO BOX                 34
PG.CEEE                          30
PG.REP.ELEVADOR                  24
PG.REMESSA DOCTOS.               22
PG.SALÁRIO                       21
VALOR                            21
PG.ADTO.SALÁRIO                  20
PG.SÍNDICO PROF.                 12
PG.REP.HIDRÁULICO                12
PG.ISSQN                         12
PG.FGTS                     

In [9]:
output_dir = Path("../exports/csv")
output_dir.mkdir(exist_ok=True)
df_extratos.to_csv(output_dir / "extratos.csv", index=False)
print(f"✓ Exportado: {len(df_extratos)} registros → exports/csv/extratos.csv")

✓ Exportado: 6215 registros → exports/csv/extratos.csv
